# Week 02 Lab: Image Processing Through a Four-Step Pipeline

**Course:** Perceptual Computing / Computer Graphics and Human-Computer Interaction

## Student Information

- **Name:** MALIHA 
- **Roll Number:** 2K24/CSE/82
- **GitHub Username:** `2K24CSE[82]`
- **GitHub Repository:** `CGHCI`

This notebook contains the five required lab tasks only.

## Problem 1: Thresholding 

### Step 1: Inputs and output
**Input:** a grayscale image and a threshold value.  
**Output:** an image containing only `0` (black) and `255` (white).

### Step 2: Rule
For every pixel:
- if `pixel >= threshold`, output `255`;
- otherwise, output `0`.

### Step 3: Manual calculation
For the test input, pixels below `128` become `0`, while pixels equal to or above `128` become `255`.

### Step 4: Python
Implement the thresholding function and run the test.

In [1]:
import numpy as np

def threshold_image(image: np.ndarray, threshold: int) -> np.ndarray:
    """Convert a grayscale image to black (0) and white (255)."""
    return np.where(image >= threshold, 255, 0).astype(np.uint8)

problem_1_input = np.array([[20, 128, 200], [100, 150, 250]], dtype=np.uint8)
problem_1_expected = np.array([[0, 255, 255], [0, 255, 255]], dtype=np.uint8)

result = threshold_image(problem_1_input, 128)
np.testing.assert_array_equal(result, problem_1_expected)

print("Problem 1 passed")
print("Output:")
print(result)


Problem 1 passed
Output:
[[  0 255 255]
 [  0 255 255]]


## Problem 2: Image Memory 

### Step 1: Inputs and output
**Inputs:** image width, image height, and bits per pixel (`bpp`).  
**Output:** uncompressed image memory in bytes.

### Step 2: Rule
```text
bytes = width × height × bpp / 8
```

### Step 3: Manual calculation
Use the required test values: `1920 × 1080` image with `24 bpp`.

### Step 4: Python
Implement the memory calculation and run the test.

In [2]:
def display_memory_bytes(width: int, height: int, bpp: int) -> int:
    """Return uncompressed image memory in bytes."""
    return (width * height * bpp) // 8

result = display_memory_bytes(1920, 1080, 24)
expected = 1920 * 1080 * 24 // 8

assert result == expected

print("Problem 2 passed")
print("Output:", result, "bytes")


Problem 2 passed
Output: 6220800 bytes


## Problem 3: Average 9 Pixels 

### Step 1: Inputs and output
**Input:** nine pixel values arranged in a `3 × 3` array.  
**Output:** the rounded average value.

### Step 2: Rule
```text
average = sum of all 9 pixel values / 9
```

### Step 3: Manual calculation
For the required test block, add all nine values and divide the sum by `9`, then round the result.

### Step 4: Python
Implement the `3 × 3` mean calculation and run the test.

In [3]:
def mean_filter_3x3(neighborhood: np.ndarray) -> int:
    """Return the rounded average of 9 pixels."""
    if neighborhood.shape != (3, 3):
        raise ValueError("Input must be a 3 x 3 array.")
    return int(round(float(np.sum(neighborhood)) / 9))

problem_3_input = np.array(
    [[10, 20, 10],
     [30, 50, 30],
     [10, 20, 10]],
    dtype=np.uint8
)

result = mean_filter_3x3(problem_3_input)
assert result == 21

print("Problem 3 passed")
print("Output:", result)


Problem 3 passed
Output: 21


## Problem 4: Change Image Contrast

### Step 1: Inputs and output
**Input:** image values, an old input range, and a new output range.  
**Output:** the values remapped to the new range.

### Step 2: Rule
```text
new = (value - old_min) / (old_max - old_min)
      × (new_max - new_min) + new_min
```

The result is clipped to the requested output range.

### Step 3: Manual calculation
Use the required test values and remap them from the old range to the new range.

### Step 4: Python
Implement the contrast-stretching calculation and run the test.

In [4]:
def contrast_stretch(
    image: np.ndarray,
    in_min: float,
    in_max: float,
    out_min: float = 0.0,
    out_max: float = 255.0,
) -> np.ndarray:
    """Change image values from one range to another and clip the result."""
    if in_max == in_min:
        raise ValueError("Input range must have different minimum and maximum values.")

    image_float = image.astype(np.float32)
    stretched = (
        (image_float - in_min) / (in_max - in_min)
        * (out_max - out_min)
        + out_min
    )
    return np.clip(stretched, out_min, out_max).astype(np.float32)

problem_4_input = np.array([50, 100, 150], dtype=np.float32)

result = contrast_stretch(problem_4_input, 50, 150)
expected = np.array([0.0, 127.5, 255.0], dtype=np.float32)

np.testing.assert_allclose(result, expected)

print("Problem 4 passed")
print("Output:", result)


Problem 4 passed
Output: [  0.  127.5 255. ]


## Problem 5: Find an Edge with Sobel 

### Step 1: Inputs and output
**Input:** a `3 × 3` grayscale image block.  
**Output:** `gx`, `gy`, and edge strength.

### Step 2: Rule
Use the Sobel kernels:

```text
Gx =       Gy =
-1  0  1   -1 -2 -1
-2  0  2    0  0  0
-1  0  1    1  2  1
```

Then calculate:

```text
strength = sqrt(gx² + gy²)
```

### Step 3: Manual calculation
Apply the two kernels to the required `3 × 3` test block and calculate the resulting edge strength.

### Step 4: Python
Implement the Sobel calculation and run the test.

In [5]:
def sobel_response(block: np.ndarray) -> tuple[float, float, float]:
    """Return gx, gy, and Sobel edge strength for a 3 x 3 block."""
    if block.shape != (3, 3):
        raise ValueError("Input must be a 3 x 3 array.")

    gx_kernel = np.array([
        [-1, 0, 1],
        [-2, 0, 2],
        [-1, 0, 1]
    ], dtype=np.float32)

    gy_kernel = np.array([
        [-1, -2, -1],
        [0, 0, 0],
        [1, 2, 1]
    ], dtype=np.float32)

    gx = float(np.sum(block * gx_kernel))
    gy = float(np.sum(block * gy_kernel))
    strength = float(np.sqrt(gx * gx + gy * gy))
    return gx, gy, strength

problem_5_input = np.array(
    [[20, 20, 200],
     [20, 20, 200],
     [20, 20, 200]],
    dtype=np.float32
)

gx, gy, strength = sobel_response(problem_5_input)

assert gx == 720.0
assert gy == 0.0
assert strength == 720.0

print("Problem 5 passed")
print("Output:")
print("gx =", gx)
print("gy =", gy)
print("strength =", strength)


Problem 5 passed
Output:
gx = 720.0
gy = 0.0
strength = 720.0
